In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import random
from gurobipy import Model as GurobiModel, GRB
from gurobipy import quicksum


# =========================
# Parameters
# =========================
PARAMS = {
    'P': 10,               # number of caissons
    'T': 60,               # horizon
    'Cmax': 5,             # max tasks in one cluster
    'window_size': 10,     # rolling window size
    'advanced': 3,         # max advance time
    'alpha': 5,            # unit cost of advancing one day
    'beta': 10,            # unit cost of delaying one day
    'Cs': 100              # setup cost
}

random.seed(42)
alpha_values = [round(random.uniform(0.01, 0.05), 6) for _ in range(PARAMS['P'])]

random.seed(42)
initial_p_values = [round(random.uniform(3.2, 3.5), 2) for _ in range(PARAMS['P'])]

print("initial alpha values:", alpha_values)
print("initial p values:", initial_p_values)


# =========================
# Extract maintenance period
# =========================
def calculate_maintenance_period_from_data(df, d_min_ratio=0.6, d_max_ratio=1.4):
    maintenance_periods = {}
    D_min_dict = {}
    D_max_dict = {}

    print("\n===== 从数据中提取维修周期 =====")
    print(f"{'Caisson':<10} {'间隔数':<10} {'间隔范围':<20} {'周期':<12} {'D_min':<10} {'D_max':<10}")
    print("-" * 80)

    for caisson_id in sorted(df['CSEM'].unique()):
        caisson_tasks = df[
            (df['CSEM'] == caisson_id) &
            (df['OPERATION'] == 'x')
        ].sort_values('DATE')

        if len(caisson_tasks) < 2:
            default_period = PARAMS['window_size']
            maintenance_periods[caisson_id] = default_period
            D_min_dict[caisson_id] = int(np.ceil(default_period * d_min_ratio))
            D_max_dict[caisson_id] = int(np.ceil(default_period * d_max_ratio))
            print(f"{caisson_id:<10} {0:<10} {'-':<20} {default_period:<12.2f} {D_min_dict[caisson_id]:<10} {D_max_dict[caisson_id]:<10}")
            continue

        intervals = []
        for i in range(1, len(caisson_tasks)):
            interval = caisson_tasks.iloc[i]['DATE'] - caisson_tasks.iloc[i - 1]['DATE']
            intervals.append(interval)

        maintenance_period = np.mean(intervals)
        maintenance_periods[caisson_id] = maintenance_period
        D_min_dict[caisson_id] = int(np.ceil(maintenance_period * d_min_ratio))
        D_max_dict[caisson_id] = int(np.ceil(maintenance_period * d_max_ratio))

        interval_min = min(intervals)
        interval_max = max(intervals)
        interval_range = f"[{interval_min}, {interval_max}]"

        print(f"{caisson_id:<10} {len(intervals):<10} {interval_range:<20} "
              f"{maintenance_period:<12.2f} {D_min_dict[caisson_id]:<10} {D_max_dict[caisson_id]:<10}")

    print("===== 提取完成 =====\n")
    return maintenance_periods, D_min_dict, D_max_dict


# =========================
# Gurobi clustering model
# =========================
def clustering_CR(df_window, window_size, window_start, global_task_count,
                  full_df, D_min_dict, D_max_dict, state):
    if df_window.empty or window_size < 1:
        print(f"warning: window {window_start} has no tasks or invalid window size")
        return df_window

    N = df_window.shape[0]
    mdl = GurobiModel('Dynamic_Window_Clustering')
    mdl.setParam('OutputFlag', 0)

    data_pos_num = window_size + 1

    theta = mdl.addVars(N, data_pos_num, vtype=GRB.BINARY, name='theta')
    CS = mdl.addVars(data_pos_num, lb=0.0, vtype=GRB.CONTINUOUS, name='CS')
    y = mdl.addVars(data_pos_num, vtype=GRB.BINARY, name='y')

    a = [0] * N
    b = [0] * N

    # each task assigned to exactly one cluster
    for n in range(N):
        mdl.addConstr(theta.sum(n, '*') == 1, name=f"assign_task_{n}")

    # cluster capacity
    for t in range(data_pos_num):
        mdl.addConstr(theta.sum('*', t) <= PARAMS['Cmax'], name=f"cap_{t}")

    # feasible time range for each task
    for n in range(N):
        local_time = int(df_window.iloc[n]['window_time'])
        caisson = int(df_window.iloc[n]['CSEM'])
        current_alpha = state[caisson]['alpha']

        a[n] = int(max(0, local_time - PARAMS['advanced']))

        ub = local_time + int((3.2 - 3.0) / max(current_alpha, 1e-9)) - 2
        b[n] = int(min(data_pos_num - 1, max(a[n], ub)))

    # range constraints
    for n in range(N):
        for t in range(data_pos_num):
            mdl.addConstr(
                t * theta[n, t] + a[n] * (1 - theta[n, t]) >= a[n],
                name=f"lb_range_n{n}_t{t}"
            )
            mdl.addConstr(
                t * theta[n, t] + b[n] * (1 - theta[n, t]) <= b[n],
                name=f"ub_range_n{n}_t{t}"
            )

    # y[t] indicates whether cluster t is used
    M = PARAMS['Cmax']
    for t in range(data_pos_num):
        task_count = theta.sum('*', t)
        mdl.addConstr(task_count + M * (1 - y[t]) >= 1, name=f"activate_lb_{t}")
        mdl.addConstr(task_count <= M * y[t], name=f"activate_ub_{t}")

    # setup cost
    setup_cost = PARAMS['Cs'] * (theta.sum() - y.sum())

    # deviation cost
    cost_matrix = {}
    for n in range(N):
        local_time_n = int(df_window.iloc[n]['window_time'])
        for t in range(data_pos_num):
            deviation = t - local_time_n
            if deviation > 0:
                cost_matrix[(n, t)] = PARAMS['beta'] * deviation
            elif deviation < 0:
                cost_matrix[(n, t)] = PARAMS['alpha'] * (-deviation)
            else:
                cost_matrix[(n, t)] = 0

    CP_total = quicksum(
        cost_matrix[(n, t)] * theta[n, t]
        for n in range(N)
        for t in range(data_pos_num)
    )

    # maintenance spacing constraints
    for n in range(N):
        row = df_window.iloc[n]
        caisson = int(row['CSEM'])

        if window_start == 0:
            continue

        prev_rows = full_df[
            (full_df['CSEM'] == caisson) &
            (full_df['CENTER'] < row['DATE']) &
            (full_df['OPERATION'] == 'x')
        ].sort_values('CENTER')

        if prev_rows.empty:
            continue

        prev_center = int(prev_rows.iloc[-1]['CENTER'])
        t_now_local = quicksum(t * theta[n, t] for t in range(data_pos_num))

        d_min_i = D_min_dict[caisson]
        d_max_i = D_max_dict[caisson]

        mdl.addConstr(
            t_now_local + window_start - prev_center >= d_min_i,
            name=f"dmin_n{n}"
        )
        mdl.addConstr(
            t_now_local + window_start - prev_center <= d_max_i,
            name=f"dmax_n{n}"
        )

    # objective
    obj = setup_cost - CP_total
    mdl.setObjective(obj, GRB.MAXIMIZE)
    mdl.optimize()

    if mdl.status not in [GRB.OPTIMAL, GRB.TIME_LIMIT, GRB.SUBOPTIMAL]:
        print(f"Failed to solve the problem: status {mdl.status}")
        return None

    centers = {}
    for n in range(N):
        for t in range(data_pos_num):
            if theta[n, t].X is not None and round(theta[n, t].X) == 1:
                centers[n] = t + window_start
                break

    index_n_list = df_window.INDEX.tolist()
    for n in range(len(index_n_list)):
        for t in range(data_pos_num):
            if round(theta[n, t].X) == 1:
                df_window.loc[df_window['INDEX'] == index_n_list[n], 'CENTER'] = centers.get(n, 0)

    print(f"Model stats: {mdl.NumVars} vars, {mdl.NumConstrs} cons")
    print(f"Objective value: {mdl.ObjVal}")

    return df_window


# =========================
# Update actual state after each window
# =========================
def propagate_states_in_window(df, state, window_start, window_end,
                               delta_low=0.05, delta_high=0.15, seed=None):
    """
    For each caisson:
    - start from state at window_start: p, alpha
    - move linearly with current alpha until each maintenance CENTER
    - at each maintenance:
        p jumps to 3.5
        alpha <- alpha * (1 + delta), delta ~ U(delta_low, delta_high)
    - continue to window_end
    """
    rng = random.Random(seed) if seed is not None else random
    updated_state = {k: v.copy() for k, v in state.items()}
    event_log = []

    for caisson in sorted(updated_state.keys()):
        s = updated_state[caisson]
        p = float(s['p'])
        alpha = float(s['alpha'])
        t_cur = max(int(s.get('last_time', 0)), window_start)

        tasks = df[
            (df['CSEM'] == caisson) &
            (df['OPERATION'] == 'x') &
            (df['CENTER'] >= window_start) &
            (df['CENTER'] <= window_end)
        ].sort_values('CENTER')

        for _, row in tasks.iterrows():
            t_task = int(row['CENTER'])
            dt = t_task - t_cur

            if dt > 0:
                p = p - alpha * dt

            p_before_fill = p
            p = 3.5

            delta = rng.uniform(delta_low, delta_high)
            alpha = alpha * (1 + delta)

            event_log.append({
                'CSEM': caisson,
                'time': t_task,
                'p_before_fill': p_before_fill,
                'p_after_fill': p,
                'alpha_before': s['alpha'] if t_cur == max(int(s.get('last_time', 0)), window_start) else None,
                'alpha_after': alpha,
                'delta': delta,
                'window_start': window_start,
                'window_end': window_end
            })

            t_cur = t_task

        dt_end = window_end - t_cur
        if dt_end > 0:
            p = p - alpha * dt_end

        updated_state[caisson]['p'] = p
        updated_state[caisson]['alpha'] = alpha
        updated_state[caisson]['last_time'] = window_end

    return updated_state, pd.DataFrame(event_log)


def state_to_df(state, window_end):
    rows = []
    for c, s in state.items():
        rows.append({
            'window_end': window_end,
            'CSEM': c,
            'p_end': s['p'],
            'alpha_end': s['alpha'],
            'last_time': s['last_time']
        })
    return pd.DataFrame(rows)


# =========================
# Update second task in each window
# =========================
def update_secondary_tasks(main_df, clustered_df, window_start, window_end):
    for caisson in clustered_df["CSEM"].unique():
        tasks = main_df[
            (main_df['CSEM'] == caisson) &
            (main_df['DATE'].between(window_start, window_end)) &
            (main_df['OPERATION'] == 'x')
        ].sort_values('DATE')

        if len(tasks) >= 2:
            first_task = tasks.iloc[0]
            second_idx = tasks.index[1]

            orig_first = first_task['DATE']
            first_center = clustered_df.loc[clustered_df['CSEM'] == caisson, 'CENTER'].iloc[0]
            move_delta = first_center - orig_first

            orig_second = tasks.iloc[1]['DATE']
            new_date = min(window_end, orig_second + move_delta)

            main_df.loc[second_idx, ['DATE', 'CENTER']] = new_date, new_date
            print(f"Caisson {caisson}: first {orig_first}->{first_center}, second {orig_second}->{new_date}")


# =========================
# Main rolling horizon
# =========================
def main(window_size):
    df = pd.read_csv('maintenance_plan_new.csv')
    df_init = df.copy()
    df_history_list = [df_init]

    sliding_windows_starts = []
    sliding_windows_ends = []

    # initialize CENTER
    df['CENTER'] = df['DATE']

    # derive maintenance spacing bounds
    maintenance_periods, D_min_dict, D_max_dict = calculate_maintenance_period_from_data(
        df=df,
        d_min_ratio=0.5,
        d_max_ratio=1.5
    )
    PARAMS['D_min'] = D_min_dict
    PARAMS['D_max'] = D_max_dict

    # dynamic state for each caisson
    state = {
        c: {
            'p': initial_p_values[c],
            'alpha': alpha_values[c],
            'last_time': 0
        }
        for c in range(PARAMS['P'])
    }

    state_history = []
    event_logs = []
    window_stats = []

    global_task_count = [0] * (PARAMS['T'] * 2)
    window_start = 0
    iter_id = 0

    while window_start <= PARAMS['T']:
        window_end = min(window_start + window_size, PARAMS['T'])

        sliding_windows_starts.append(window_start)
        sliding_windows_ends.append(window_end)

        mask = (
            (df['DATE'] >= window_start) &
            (df['DATE'] <= window_end) &
            (df['OPERATION'] == 'x')
        )

        window_df = df.loc[mask].copy()

        if window_df.empty:
            if window_end >= PARAMS['T']:
                break
            window_start = window_end
            continue

        window_df['window_time'] = window_df['DATE'] - window_start

        first_tasks = window_df.groupby('CSEM', group_keys=False).apply(
            lambda x: x.sort_values('DATE').iloc[0]
        )

        clustered_df = clustering_CR(
            first_tasks,
            window_size,
            window_start,
            global_task_count,
            df,
            D_min_dict,
            D_max_dict,
            state
        )

        center_map = None

        if clustered_df is not None:
            center_map = clustered_df.set_index('INDEX')['CENTER']
            df.loc[df['INDEX'].isin(center_map.index), 'CENTER'] = df['INDEX'].map(center_map)

            update_secondary_tasks(df, clustered_df, window_start, window_end)

            state, event_log = propagate_states_in_window(
                df=df,
                state=state,
                window_start=window_start,
                window_end=window_end,
                delta_low=0.05,
                delta_high=0.15,
                seed=1000 + iter_id
            )

            state_history.append(state_to_df(state, window_end))
            event_logs.append(event_log)

            window_stats.append({
                'iter': iter_id,
                'window_start': window_start,
                'window_end': window_end
            })

        # move to next rolling window
        second_dates = []
        if clustered_df is not None:
            for caisson in clustered_df["CSEM"].unique():
                tasks = df[
                    (df['CSEM'] == caisson) &
                    (df['OPERATION'] == 'x') &
                    (df['DATE'].between(window_start, window_end))
                ].sort_values('DATE')

                if len(tasks) > 1:
                    second_dates.append(tasks.iloc[1]['DATE'])

        if second_dates:
            window_start = min(second_dates)
        else:
            window_start = window_start + window_size

        df_history_list.append(df.copy())
        iter_id += 1

        if window_end >= PARAMS['T']:
            break

        if center_map is not None:
            df.loc[df['INDEX'].isin(center_map.index), 'DATE'] = df.loc[df['INDEX'].isin(center_map.index), 'INDEX'].map(center_map)

    # y tasks are not clustered
    df.loc[df['OPERATION'] == "y", 'CENTER'] = df.loc[df['OPERATION'] == "y", 'DATE']

    # remaining x tasks
    mask_remain = (df['OPERATION'] == "x") & (df['CENTER'] == 0)
    df.loc[mask_remain, 'CENTER'] = df.loc[mask_remain, 'DATE']

    # final spacing diagnostic
    df_sorted = df[df['OPERATION'] == 'x'].sort_values(['CSEM', 'CENTER'])
    df_sorted['PREV_CENTER'] = df_sorted.groupby('CSEM')['CENTER'].shift(1)
    df_sorted['GAP'] = df_sorted['CENTER'] - df_sorted['PREV_CENTER']

    print("\n===== 最终任务间隔分析 =====")
    print(df_sorted[['CSEM', 'CENTER', 'PREV_CENTER', 'GAP']].head(20))
    print(f"gap min: {df_sorted['GAP'].min()}, gap max: {df_sorted['GAP'].max()}")
    print("===== 分析结束 =====\n")

    state_history_df = pd.concat(state_history, ignore_index=True) if state_history else pd.DataFrame()
    event_log_df = pd.concat(event_logs, ignore_index=True) if event_logs else pd.DataFrame()
    window_stats_df = pd.DataFrame(window_stats)

    return (
        df,
        df_history_list,
        sliding_windows_starts,
        sliding_windows_ends,
        state_history_df,
        event_log_df,
        window_stats_df
    )


# =========================
# Run
# =========================
result_df, df_history_list, sliding_windows_starts, sliding_windows_ends, state_history_df, event_log_df, window_stats_df = main(PARAMS['window_size'])

result_df.to_csv("clustering_result.csv", index=False)
state_history_df.to_csv("state_history.csv", index=False)
event_log_df.to_csv("event_log.csv", index=False)
window_stats_df.to_csv("window_stats.csv", index=False)

print("Done.")
print("Files saved:")
print("- clustering_result.csv")
print("- state_history.csv")
print("- event_log.csv")
print("- window_stats.csv")